# 05 · Evaluación retrospectiva y modelos servidos

Explica el resultado histórico y verifica las probabilidades de los dos modelos tempranos exportados. No se cambia la selección ni se presenta 2014J como prueba nueva.

**Alcance de validación.** 2014J ya se había observado en una versión anterior. **No es una prueba final independiente ni un hold-out de un solo uso.** Separar personas y ajustar transformaciones dentro de cada fold evita fugas directas, pero no elimina el conocimiento previo de sus resultados. Las cifras sobre 2014J describen este histórico; la decisión institucional necesita una cohorte futura sin consultar, con protocolo fijado antes de conocer resultados.

In [1]:
from pathlib import Path
import os, sys, json, hashlib
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'src/notebooks.py').is_file()), None)
if ROOT is None and (here/'proyecto_final/src/notebooks.py').is_file():
    ROOT = here/'proyecto_final'
assert ROOT is not None, 'Abrir desde proyecto_final o notebooks.'
sys.path.insert(0, str(ROOT))
os.environ['MPLCONFIGDIR'] = str(ROOT/'.local/matplotlib')
os.environ['PSET4_MODEL_MODE'] = 'fixture'
from IPython.display import display, Markdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.notebooks import evidencia
from src.datos import sha256
E = evidencia()
pd.set_option('display.max_columns', 12)
pd.set_option('display.max_rows', 32)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
%matplotlib inline
print('Ejecución local de evidencia congelada. Sin llamadas LLM, nuevos entrenamientos ni cambios en Registry.')


Ejecución local de evidencia congelada. Sin llamadas LLM, nuevos entrenamientos ni cambios en Registry.


## Evaluación histórica y limitación de independencia

El resultado del día 56 pertenece al estudio previo. Aunque HGB y Logística se compararon con igual capacidad, conocer 2014J impide interpretar esa evaluación como una confirmación final independiente.

In [2]:
result=E['seleccion']; assert result['test_previously_observed_in_v1'] is True
assert result['winner_changed_using_test'] is False
rows=[]
for name,r in result['models'].items():
    rows.append({'rol':name,'familia':r['family'],**r['technical'],**r['business']['0.1']})
display(pd.DataFrame(rows))
display(pd.DataFrame([result['longitudinal_vs_matched_static']]))
print('Run final histórico día 56:',result['final_run_id'])
print('Consecuencia: validar con una cohorte futura sin consultar antes de adoptar o ampliar el uso.')


,rol,familia,ap_fail,logloss,brier_fail,tp,...,n,fail,k,coverage,precision_at_k,unique_students_selected
0,winner,boosting,0.426507,0.842232,0.159718,465,...,8758,1947,872,0.238829,0.533257,845
1,challenger,lineal,0.438755,0.935294,0.159743,477,...,8758,1947,872,0.244992,0.547018,846
2,static_boosting,boosting,0.310455,0.925204,0.171608,309,...,8758,1947,872,0.158706,0.354358,857
3,static_lineal,lineal,0.315410,0.916969,0.170393,323,...,8758,1947,872,0.165896,0.370413,854
4,dummy,dummy,0.222311,13.889853,0.222311,174,...,8758,1947,872,0.089368,0.199541,860


,static_tp,longitudinal_tp,absolute_tp_gain,relative_tp_gain_pct,causal_effect
0,309,465,156,50.485437,False


Run final histórico día 56: fcd23ae22d6c46ffb41c1e4f23e1c02d
Consecuencia: validar con una cohorte futura sin consultar antes de adoptar o ampliar el uso.


## Semanas 2 y 4: modelos distintos del día 56

Se reproducen probabilidades y AP sobre las muestras de desarrollo incluidas en las fixtures. Esta comprobación acredita portabilidad del artefacto; no es una medición de generalización ni la AP reportada en validación cruzada.

In [3]:
import mlflow.sklearn
from threadpoolctl import threadpool_limits
from sklearn.metrics import average_precision_score
from src.pipeline import CATEGORICAS
ref=json.loads((ROOT/'tests/fixtures/referencia.json').read_text()); rows=[]
for cut in [14,28]:
    sample=json.loads((ROOT/f'tests/fixtures/muestra_semana{cut//7}.json').read_text())
    x=pd.DataFrame([r['features'] for r in sample['rows']])
    for c in x:
        x[c]=x[c].astype(object).where(x[c].notna(),np.nan) if c in CATEGORICAS else x[c].astype(float)
    model=mlflow.sklearn.load_model(str(ROOT/f'tests/fixtures/semana{cut//7}'))
    with threadpool_limits(limits=2): probabilities=model.predict_proba(x)[:,list(model.classes_).index('Fail')]
    np.testing.assert_allclose(probabilities,sample['expected_p_fail'],rtol=0,atol=ref['tolerance'])
    ap=average_precision_score(np.asarray([r['target'] for r in sample['rows']])=='Fail', probabilities)
    np.testing.assert_allclose(ap,ref['modelos'][str(cut)]['ci_ap_fail'],rtol=0,atol=ref['tolerance'])
    rows.append({'corte':cut,'filas_fixture':len(x),'AP_solo_muestra_desarrollo':ap,'probabilidades_reproducidas':True})
display(pd.DataFrame(rows))
early=E['anticipacion']['models']
active=['a6c0b65d4e354ceaabfc565c88bfa0c2','048df6920ba747d28d957b052e685666']
display(pd.DataFrame([{k:r[k] for k in ['run_id','ap_fail_mean','coverage_mean']} for r in early if r['run_id'] in active]))


,corte,filas_fixture,AP_solo_muestra_desarrollo,probabilidades_reproducidas
0,14,60,0.600065,True
1,28,60,0.533479,True


,run_id,ap_fail_mean,coverage_mean
0,a6c0b65d4e354ceaabfc565c88bfa0c2,0.456963,0.186278
1,048df6920ba747d28d957b052e685666,0.492690,0.201457


## Lectura final

Los resultados reproducidos pertenecen a los experimentos registrados. La fecha de ejecución de este notebook no los convierte en evidencia nueva de generalización. Consulte `README.md` para insumos, dependencias y comandos del pipeline completo.